# apply_style_00

この notebook は、既存コードを再利用しながら「前景/背景を分けて style を適用する」検証用です。後で 1 つの .py にまとめやすい構成を意識して、処理を関数分割しています。

## 目的
1. GroundingDINO + SAM で前景と背景を分離する
2. 前景と背景にそれぞれ apply_style を適用する（ここでは oil_painting）

## 実装方針
- 既存コードは import して再利用する
- 既存にない機能は notebook 側で実装する

## この notebook で再利用している既存実装
- `postprocess.detectors` から GroundingDINO/SAM 関連関数を import
- `postprocess.apply_style_ver5` から `apply_style_frame` を import
- `utils.video_utility` から動画の読み書き/比較表示関数を import

## notebook 側で実装する処理
1. DINO+SAM で前景マスク（0/255）を生成
2. マスクで前景画像と背景画像を分離
3. 前景・背景をそれぞれ `apply_style_frame` で stylize
4. ソフトマスクで合成して最終フレームを作成
5. これを動画全フレームへ適用

## セル構成
- Cell 2: import と環境準備
- Cell 3: 前景マスク生成と前景/背景分離
- Cell 4: 前景/背景 style 適用と動画処理関数
- Cell 5: 単一フレーム確認（任意）
- Cell 6: 動画実行

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import cv2
import numpy as np
from tqdm import tqdm

sys.path.append('/workspace/src')

from postprocess.detectors import detect_all_boxes, get_sam_mask_from_box
from postprocess.apply_style_ver5 import apply_style_frame
from utils.video_utility import load_video, write_video, show_before_after

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_foreground_mask_dino_sam(
    frame_bgr: np.ndarray,
    text_prompt: str = 'person .',
    box_threshold: float = 0.3,
    text_threshold: float = 0.25,
    dilate_iter: int = 1,
    close_iter: int = 1,
) -> np.ndarray:
    """GroundingDINO + SAM で前景マスク(0/255)を作る。"""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    boxes = detect_all_boxes(
        frame_rgb,
        text_prompt=text_prompt,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
    )

    h, w = frame_bgr.shape[:2]
    fg_mask = np.zeros((h, w), dtype=np.uint8)

    for box in boxes:
        sam_mask = get_sam_mask_from_box(frame_rgb, box)
        if sam_mask is None:
            continue
        fg_mask = np.maximum(fg_mask, (sam_mask > 0).astype(np.uint8) * 255)

    if close_iter > 0:
        kernel = np.ones((3, 3), dtype=np.uint8)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)
    if dilate_iter > 0:
        kernel = np.ones((3, 3), dtype=np.uint8)
        fg_mask = cv2.dilate(fg_mask, kernel, iterations=dilate_iter)

    return fg_mask


def split_foreground_background(
    frame_bgr: np.ndarray,
    fg_mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """前景画像と背景画像に分割する。"""
    mask3 = (fg_mask > 0)[:, :, None]
    foreground = np.where(mask3, frame_bgr, 0).astype(np.uint8)
    background = np.where(~mask3, frame_bgr, 0).astype(np.uint8)
    return foreground, background

In [3]:
def apply_style_foreground_background(
    frame_bgr: np.ndarray,
    style: str = 'oil_painting',
    text_prompt: str = 'person .',
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    1) DINO+SAM で前景マスク生成
    2) 前景/背景に分割
    3) 前景・背景それぞれに apply_style を実行
    4) 合成して最終フレームを返す
    """
    fg_mask = build_foreground_mask_dino_sam(frame_bgr, text_prompt=text_prompt)
    fg, bg = split_foreground_background(frame_bgr, fg_mask)

    stylized_fg = apply_style_frame(fg, style)
    stylized_bg = apply_style_frame(bg, style)

    alpha = cv2.GaussianBlur((fg_mask.astype(np.float32) / 255.0), (0, 0), 1.2)
    alpha3 = alpha[:, :, None]
    merged = (stylized_fg.astype(np.float32) * alpha3 + stylized_bg.astype(np.float32) * (1.0 - alpha3))
    merged = np.clip(merged, 0, 255).astype(np.uint8)

    return merged, fg_mask, fg, bg, stylized_fg


def apply_style_video_foreground_background(
    in_path: str | Path,
    out_path: str | Path,
    style: str = 'oil_painting',
    text_prompt: str = 'person .',
    max_frames: int | None = None,
) -> Path:
    frames, fps, width, height = load_video(in_path)

    if max_frames is not None:
        frames = frames[:max_frames]

    out_frames: list[np.ndarray] = []
    for frame in tqdm(frames, desc=f'fg/bg apply_style ({style})'):
        merged, _mask, _fg, _bg, _stylized_fg = apply_style_foreground_background(
            frame,
            style=style,
            text_prompt=text_prompt,
        )
        out_frames.append(merged)

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    write_video(out_path, out_frames, fps, width, height)
    return out_path

In [4]:
# 単一フレーム確認（任意）
WORKSPACE = Path('/workspace')
VIDEO_DIR = WORKSPACE / 'data' / 'videos'

VIDEO_NAME = '94msufYZzaQ_26_0to273.mp4'
STYLE = 'oil_painting'
TARGET_PROMPT = 'person .'

frames, fps, w, h = load_video(VIDEO_DIR / VIDEO_NAME)
test_frame = frames[0]

merged, fg_mask, fg, bg, stylized_fg = apply_style_foreground_background(
    test_frame,
    style=STYLE,
    text_prompt=TARGET_PROMPT,
)

print('frame shape:', test_frame.shape)
print('mask ratio :', float((fg_mask > 0).sum()) / float(fg_mask.size))

final text_encoder_type: bert-base-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2274.38it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 207.50it/s] 7.48s/it]
CLIPTextModel LOAD REPORT from: /workspace/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113

frame shape: (1080, 1920, 3)
mask ratio : 0.48277922453703703


In [5]:
# 動画実行
WORKSPACE = Path('/workspace')
VIDEO_DIR = WORKSPACE / 'data' / 'videos'
OUT_DIR = WORKSPACE / 'logs' / 'notebook' / 'task_03_apply_style_fg_bg'
OUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_NAME = '94msufYZzaQ_26_0to273.mp4'
STYLE = 'oil_painting'
TARGET_PROMPT = 'person .'
MAX_FRAMES = 30 # None  # デバッグ時は 24 などに変更

in_path = VIDEO_DIR / VIDEO_NAME
out_path = OUT_DIR / f'processed_fg_bg_{STYLE}_{VIDEO_NAME}'

if not in_path.exists():
    raise FileNotFoundError(f'input video not found: {in_path}')

saved_path = apply_style_video_foreground_background(
    in_path=in_path,
    out_path=out_path,
    style=STYLE,
    text_prompt=TARGET_PROMPT,
    max_frames=MAX_FRAMES,
)

print('input :', in_path)
print('style :', STYLE)
print('prompt:', TARGET_PROMPT)
print('out   :', saved_path)

show_before_after(in_path, saved_path)

fg/bg apply_style (oil_painting): 100%|██████████| 30/30 [06:27<00:00, 12.93s/it]


input : /workspace/data/videos/94msufYZzaQ_26_0to273.mp4
style : oil_painting
prompt: person .
out   : /workspace/logs/notebook/task_03_apply_style_fg_bg/processed_fg_bg_oil_painting_94msufYZzaQ_26_0to273.mp4
